<a href="https://colab.research.google.com/github/natan-costa01/modelo_classificao/blob/main/machine_learning_SERS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##MODELO DE CLASSIFICAÇÃO

Treinar um modelo preditivo de classificação utilizando o algoritimo de Regressão Logística (Logistic Regression do sklearn) para prever se a rede elétrica está instável ou estável com base nos atributos fornecidos na fonte:

https://archive.ics.uci.edu/dataset/471/electrical+grid+stability+simulated+data

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#Algoritimo de regressão logística
from sklearn.linear_model import LogisticRegression

#Função que permite separação de treino e teste
from sklearn.model_selection import train_test_split

#Métricas para classificação
from sklearn.metrics import accuracy_score

#Matriz de confusão
from sklearn.metrics import confusion_matrix

In [10]:
#Carregar os dados e criar o DF

df = pd.read_csv('https://raw.githubusercontent.com/natan-costa01/modelo_classificao/refs/heads/main/Data_for_UCI_named.csv')
df.head(10)

,tau1,tau2,tau3,tau4,p1,p2,p3,p4,g1,g2,g3,g4,stab,stabf
0,2.959060,3.079885,8.381025,9.780754,3.763085,-0.782604,-1.257395,-1.723086,0.650456,0.859578,0.887445,0.958034,0.055347,unstable
1,9.304097,4.902524,3.047541,1.369357,5.067812,-1.940058,-1.872742,-1.255012,0.413441,0.862414,0.562139,0.781760,-0.005957,stable
2,8.971707,8.848428,3.046479,1.214518,3.405158,-1.207456,-1.277210,-0.920492,0.163041,0.766689,0.839444,0.109853,0.003471,unstable
3,0.716415,7.669600,4.486641,2.340563,3.963791,-1.027473,-1.938944,-0.997374,0.446209,0.976744,0.929381,0.362718,0.028871,unstable
4,3.134112,7.608772,4.943759,9.857573,3.525811,-1.125531,-1.845975,-0.554305,0.797110,0.455450,0.656947,0.820923,0.049860,unstable
5,6.999209,9.109247,3.784066,4.267788,4.429669,-1.857139,-0.670397,-1.902133,0.261793,0.077930,0.542884,0.469931,-0.017385,stable
6,6.710166,3.765204,6.929314,8.818562,2.397419,-0.614590,-1.208826,-0.574004,0.177890,0.397977,0.402046,0.376630,0.005954,unstable
7,6.953512,1.379125,5.719400,7.870307,3.224495,-0.748998,-1.186517,-1.288980,0.371385,0.633204,0.732741,0.380544,0.016634,unstable
8,4.689852,4.007747,1.478573,3.733787,4.041300,-1.410344,-1.238204,-1.392751,0.269708,0.250364,0.164941,0.482439,-0.038677,stable
9,9.841496,1.413822,9.769856,7.641616,4.727595,-1.991363,-0.857637,-1.878594,0.376356,0.544415,0.792039,0.116263,0.012383,unstable


In [11]:
# Inspeção básica

df.shape

(10000, 14)

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   tau1    10000 non-null  float64
 1   tau2    10000 non-null  float64
 2   tau3    10000 non-null  float64
 3   tau4    10000 non-null  float64
 4   p1      10000 non-null  float64
 5   p2      10000 non-null  float64
 6   p3      10000 non-null  float64
 7   p4      10000 non-null  float64
 8   g1      10000 non-null  float64
 9   g2      10000 non-null  float64
 10  g3      10000 non-null  float64
 11  g4      10000 non-null  float64
 12  stab    10000 non-null  float64
 13  stabf   10000 non-null  object 
dtypes: float64(13), object(1)
memory usage: 1.1+ MB


In [18]:
df.columns

Index(['tau1', 'tau2', 'tau3', 'tau4', 'p1', 'p2', 'p3', 'p4', 'g1', 'g2',
       'g3', 'g4', 'stab', 'stabf'],
      dtype='object')

In [14]:
# Estastísticas Descritivas
# Somente atributos categóricos

df.describe(include='object')

,stabf
count,10000
unique,2
top,unstable
freq,6380


In [20]:
# Quais as classes temos no atributo 'stabf'

df['stabf'].unique()

array(['unstable', 'stable'], dtype=object)

In [21]:
# Quantos registros temos de cada classe em 'stabf'

df['stabf'].value_counts()

,count
stabf,
unstable,6380
stable,3620


In [26]:
df.columns

Index(['tau1', 'tau2', 'tau3', 'tau4', 'p1', 'p2', 'p3', 'p4', 'g1', 'g2',
       'g3', 'g4', 'stab', 'stabf'],
      dtype='object')

In [28]:
# Dados de entrada X ----> Features ----> DF
# Variáveis Independentes

X = df.drop(['stab', 'stabf'], axis=1)
X.head()

,tau1,tau2,tau3,tau4,p1,p2,p3,p4,g1,g2,g3,g4
0,2.959060,3.079885,8.381025,9.780754,3.763085,-0.782604,-1.257395,-1.723086,0.650456,0.859578,0.887445,0.958034
1,9.304097,4.902524,3.047541,1.369357,5.067812,-1.940058,-1.872742,-1.255012,0.413441,0.862414,0.562139,0.781760
2,8.971707,8.848428,3.046479,1.214518,3.405158,-1.207456,-1.277210,-0.920492,0.163041,0.766689,0.839444,0.109853
3,0.716415,7.669600,4.486641,2.340563,3.963791,-1.027473,-1.938944,-0.997374,0.446209,0.976744,0.929381,0.362718
4,3.134112,7.608772,4.943759,9.857573,3.525811,-1.125531,-1.845975,-0.554305,0.797110,0.455450,0.656947,0.820923


In [37]:
df['stabf'] = df['stabf'].replace({'unstable': 'INSTÁVEL', 'stable': 'ESTÁVEL'})
df['stabf'].value_counts()

,count
stabf,
INSTÁVEL,6380
ESTÁVEL,3620


In [32]:
# Dados de saida y ----> Target ----> O que vamos prever
# Variável dependente

y = df['stabf']
y.head()

,stabf
0,unstable
1,stable
2,unstable
3,unstable
4,unstable


In [40]:
# Separação de dados para treino e para teste (train_test_split)
# X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=17)

In [41]:
X.shape[0]

10000

In [42]:
X_train.shape[0]

7000

In [43]:
X_test.shape[0]

3000

In [45]:
# Instanciando o modelo com base no algoritimo Logistic Regression

modelo = LogisticRegression()

In [46]:
# Treinando o modelo

modelo.fit(X_train, y_train)

LogisticRegression()

In [47]:
# Gerando as previsões

probabilidades = modelo.predict_proba(X_test)
probabilidades[:50]

array([[0.22408593, 0.77591407],
       [0.04586101, 0.95413899],
       [0.01003956, 0.98996044],
       [0.2024741 , 0.7975259 ],
       [0.23402843, 0.76597157],
       [0.47827435, 0.52172565],
       [0.29389383, 0.70610617],
       [0.02047682, 0.97952318],
       [0.33961632, 0.66038368],
       [0.15378889, 0.84621111],
       [0.69634298, 0.30365702],
       [0.14056611, 0.85943389],
       [0.30548179, 0.69451821],
       [0.9548015 , 0.0451985 ],
       [0.48810613, 0.51189387],
       [0.01799429, 0.98200571],
       [0.0074897 , 0.9925103 ],
       [0.43424952, 0.56575048],
       [0.23310349, 0.76689651],
       [0.14162363, 0.85837637],
       [0.03719522, 0.96280478],
       [0.83560324, 0.16439676],
       [0.02902593, 0.97097407],
       [0.74485986, 0.25514014],
       [0.0554525 , 0.9445475 ],
       [0.16829156, 0.83170844],
       [0.51976628, 0.48023372],
       [0.45121824, 0.54878176],
       [0.28275169, 0.71724831],
       [0.76443956, 0.23556044],
       [0.

In [48]:
# Gerando as previsões

previsoes = modelo.predict(X_test)
previsoes[:50]

array(['unstable', 'unstable', 'unstable', 'unstable', 'unstable',
       'unstable', 'unstable', 'unstable', 'unstable', 'unstable',
       'stable', 'unstable', 'unstable', 'stable', 'unstable', 'unstable',
       'unstable', 'unstable', 'unstable', 'unstable', 'unstable',
       'stable', 'unstable', 'stable', 'unstable', 'unstable', 'stable',
       'unstable', 'unstable', 'stable', 'stable', 'stable', 'stable',
       'unstable', 'stable', 'stable', 'unstable', 'unstable', 'unstable',
       'stable', 'unstable', 'unstable', 'unstable', 'unstable', 'stable',
       'unstable', 'stable', 'unstable', 'stable', 'unstable'],
      dtype=object)